# CTB simulation models and what-if reproduction

This notebook is the executable handover for the simulation models used in the thesis. It loads and inspects the common Inductive-Miner Petri net, the calibrated discovered parameter bundle, the intermediate rules-only workload-blind baseline, the final domain-constrained baseline, and both what-if models. It then reruns the saved experiment models with the published trace count, start time, and matched seeds and compares every generated result table with the frozen thesis outputs.

Choose **Run All**. The first cell creates an isolated `.reviewer_env` and installs the pinned packages there; it never replaces packages inside the running Jupyter kernel. The default `full` run then performs 30 simulations of 17,892 cases and took approximately 30 minutes on the author's computer. Pickle files are loaded only after their SHA-256 hashes have been verified. The reviewer runner also sorts enabled Petri-net transitions by stable transition name immediately before ProSiT's weighted draw. This preserves the model and its probabilities while making fixed seeds stable after the pickles are loaded in a new Python process.

In [ ]:
from pathlib import Path
import hashlib
import os
import subprocess
import sys
import venv

# Work whether Jupyter starts in the repository root or in this folder.
candidates = [Path.cwd(), Path.cwd() / 'reproducibility']
candidates += [parent / 'reproducibility' for parent in Path.cwd().parents]
PACKAGE_ROOT = next((p.resolve() for p in candidates if (p / 'model_manifest.json').is_file()), None)
if PACKAGE_ROOT is None:
    raise FileNotFoundError('Could not locate reproducibility/model_manifest.json')

if not ((3, 11) <= sys.version_info[:2] < (3, 13)):
    raise RuntimeError('This reproduction package requires a Python 3.11 or 3.12 Jupyter kernel.')

requirements = PACKAGE_ROOT / 'requirements.txt'
environment_root = PACKAGE_ROOT / '.reviewer_env'
environment_python = environment_root / ('Scripts/python.exe' if os.name == 'nt' else 'bin/python')
fingerprint = hashlib.sha256(requirements.read_bytes()).hexdigest()
marker = environment_root / '.requirements.sha256'
if not environment_python.is_file():
    print('Creating isolated reviewer environment ...')
    venv.create(environment_root, with_pip=True)
if not marker.is_file() or marker.read_text(encoding='utf-8').strip() != fingerprint:
    print('Installing pinned packages into the isolated reviewer environment ...')
    subprocess.check_call([str(environment_python), '-m', 'pip', 'install', '--disable-pip-version-check', '-r', str(requirements)])
    marker.write_text(fingerprint + '\n', encoding='utf-8')
else:
    print('Isolated reviewer environment is already current.')

def run_external(code):
    env = os.environ.copy()
    env['PYTHONPATH'] = str(PACKAGE_ROOT) + os.pathsep + env.get('PYTHONPATH', '')
    env['MPLBACKEND'] = 'Agg'
    completed = subprocess.run(
        [str(environment_python), '-c', code], cwd=PACKAGE_ROOT, env=env,
        text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    print(completed.stdout, end='')
    if completed.returncode:
        raise RuntimeError(f'Isolated reviewer command failed with exit code {completed.returncode}.')
    return completed.stdout

print('Package root:', PACKAGE_ROOT)
print('Isolated Python:', environment_python)

In [ ]:
import json
from IPython.display import Image, display

MODELS_DIR = PACKAGE_ROOT / 'models'
EXPECTED_DIR = PACKAGE_ROOT / 'expected_results'
OUTPUTS_DIR = PACKAGE_ROOT / 'outputs'
run_external("import reviewer_runner as rr; print(rr.package_versions().to_string(index=False))")

## 1. Verify and load the frozen files

The manifest fixes the identity and role of every model and expected-result file. The pickles are not opened until all hashes match.

In [ ]:
run_external(r"""
import reviewer_runner as rr
integrity = rr.verify_package_files()
print(integrity.to_string(index=False))
models = rr.load_models()
print('\nLoaded:', ', '.join(models))
""")

## 2. Inspect the Petri net

All four bundles must contain the same 7-place, 16-transition, 32-arc Petri net. The PNML below is also loaded independently to demonstrate a portable control-flow round trip.

In [ ]:
display(Image(filename=str(MODELS_DIR / 'ctb_inductive_miner_petri_net.png')))
run_external(r"""
import pm4py
import reviewer_runner as rr
models = rr.load_models()
print(rr.petri_net_transitions(models['baseline']).to_string(index=False))
net, initial, final = pm4py.read_pnml(str(rr.MODELS_DIR / 'ctb_inductive_miner.pnml'))
shape = (len(net.places), len(net.transitions), len(net.arcs))
print('\nPNML round trip:', shape[0], 'places,', shape[1], 'transitions,', shape[2], 'arcs')
assert shape == (7, 16, 32)
""")

## 3. Inspect the parameter bundles

`discovered_source` is the training-only calibrated result before the domain correction. `rules_only_workload_blind` is the intermediate baseline with process-state rules enabled, explicit workload features disabled, and workload proxy attributes removed before discovery. `baseline` is the effective original-process model used in the matched scenario runner. The remaining two rows are the independently derived interventions.

In [ ]:
run_external(r"""
import reviewer_runner as rr
models = rr.load_models()
summary = rr.model_summary(models)
print(summary.drop(columns=['petri_net_signature']).to_string(index=False))
print('\nExecutable model-difference contracts:')
print(rr.assert_model_contracts(models).to_string(index=False))
assert summary['petri_net_signature'].nunique() == 1
""")

In [ ]:
run_external(r"""
import reviewer_runner as rr
models = rr.load_models()
inventory = rr.parameter_component_inventory(models['baseline'])
grouped = inventory.groupby('family', as_index=False).agg(
    components=('component', 'count'), split_nodes=('split_nodes', 'sum'),
    leaf_nodes=('leaf_nodes', 'sum'), stored_runtime_samples=('stored_runtime_samples', 'sum'),
)
print(grouped.to_string(index=False))
print('\nRouting and arrival components:')
print(inventory[inventory['family'].isin(['routing', 'arrival_time'])].to_string(index=False))
print('\nExecution-time components:')
print(inventory[inventory['family'].eq('execution_time')].to_string(index=False))
""")

## 4. Inspect the interventions

Scenario A changes RMG resource eligibility and closes T22's calendar. Scenario B changes only the empirical arrival model. The detailed frozen difference contract is shown first, followed by compact executable checks.

In [ ]:
with (MODELS_DIR / 'scenario_parameter_changes.json').open(encoding='utf-8') as handle:
    changes = json.load(handle)
print(json.dumps(changes, indent=2))
run_external(r"""
import pandas as pd
import reviewer_runner as rr
models = rr.load_models()
rows = []
for activity in ('RMG_receive', 'RMG_delivery', 'RMG_mixed'):
    rows.append({
        'activity': activity,
        'baseline_resources': len(models['baseline'].act_to_resources[activity]),
        'scenario_A_resources': len(models['t22_closed'].act_to_resources[activity]),
        'removed': sorted(set(models['baseline'].act_to_resources[activity]) - set(models['t22_closed'].act_to_resources[activity])),
    })
print(pd.DataFrame(rows).to_string(index=False))
""")

## 5. Demonstrate ProSiT's documented JSON save/load API

The baseline is exported with `SimulatorParameters.to_json()` and an import is attempted with `from_json()`. For this CTB model, ProSiT 1.0.3 cannot parse empirical attribute-tuple keys containing `nan`; it also omits the per-leaf `sampled` arrays used by the CTB empirical calibration. The cell reports this API limitation instead of concealing it. The verified pickle is therefore the authoritative exact load path.

In [ ]:
run_external(r"""
import pandas as pd
import reviewer_runner as rr
models = rr.load_models()
report = rr.export_and_reload_official_json(
    models['baseline'], rr.OUTPUTS_DIR / 'baseline_parameters_official_prosit.json'
)
print(pd.Series(report, name='ProSiT JSON round trip').to_string())
assert report['json_export_succeeded']
assert not report['exact_ctb_runtime_state_restored']
""")

## 6. Reproduce the saved-model simulations

The full configuration loads the three exact executable models and runs seeds 42-51 with 17,892 cases per model and seed, starting at 2026-04-20 18:17. The same seed is used for baseline and both scenarios before advancing to the next seed. Every generated event log is checked for case boundaries, within-case overlap, Gate Out ordering, prohibited T22 assignments, case count, and duration tails.

In [ ]:
RUN_MODE = 'full'  # Change only to 'smoke' for a quick 2-seed x 250-case mechanics test.
fresh_output = OUTPUTS_DIR / f'{RUN_MODE}_reproduction'
run_external(
    "import reviewer_runner as rr; "
    f"print(rr.run_saved_models(mode={RUN_MODE!r}))"
)

## 7. Verify the replicated results

For a full run, all per-seed KPIs, structural contracts, scenario summaries, and paired confidence intervals must match the frozen thesis tables within numerical floating-point tolerance.

In [ ]:
verification_code = rf"""
from pathlib import Path
import pandas as pd
import reviewer_runner as rr
output = Path({str(fresh_output)!r})
if {RUN_MODE!r} == 'full':
    print(rr.compare_with_frozen_results(output).to_string(index=False))
    print('\nFULL REPRODUCTION PASS')
else:
    print('SMOKE PASS: mechanics and structural contracts passed; thesis KPIs were not tested.')
paired = pd.read_csv(output / 'scenario_paired_delta_summary.csv')
headline = paired[paired['metric'].isin([
    'mean_turnaround_min', 'mean_rmg_service_min',
    'mean_rmg_pre_service_min', 'arrival_rate_per_elapsed_hour',
])][['scenario', 'metric', 'mean_delta', 'ci95_delta_lo', 'ci95_delta_hi', 'ci_excludes_zero']]
print('\nHeadline paired effects:')
print(headline.to_string(index=False))
"""
run_external(verification_code)
display(Image(filename=str(fresh_output / 'figures' / 'scenario_paired_deltas_ci.png')))

## Interpretation boundary

A passing notebook establishes that the stored models are inspectable, executable, structurally valid, and numerically reproducible under the pinned software environment. It does not turn the what-if changes into validated physical counterfactuals: the models do not represent container relocation, crane travel, lane geometry, or explicit spatial queues.